In [ ]:
# Testing to see if nfl_data_py is a good dataset

import nfl_data_py as nfl
import pandas as pd

years = list(range(2015, 2026))  # 2015 through 2025
schedules = nfl.import_schedules(years)
print(schedules.shape)
schedules.head()

(3028, 46)


,game_id,season,game_type,week,gameday,weekday,gametime,away_team,away_score,home_team,...,wind,away_qb_id,home_qb_id,away_qb_name,home_qb_name,away_coach,home_coach,referee,stadium_id,stadium
4248,2015_01_PIT_NE,2015,REG,1,2015-09-10,Thursday,20:30,PIT,21.0,NE,...,7.0,00-0022924,00-0019596,Ben Roethlisberger,Tom Brady,Mike Tomlin,Bill Belichick,Carl Cheffers,BOS00,Gillette Stadium
4249,2015_01_IND_BUF,2015,REG,1,2015-09-13,Sunday,13:00,IND,14.0,BUF,...,15.0,00-0029668,00-0028118,Andrew Luck,Tyrod Taylor,Chuck Pagano,Rex Ryan,John Parry,BUF00,Ralph Wilson Stadium
4250,2015_01_GB_CHI,2015,REG,1,2015-09-13,Sunday,13:00,GB,31.0,CHI,...,11.0,00-0023459,00-0024226,Aaron Rodgers,Jay Cutler,Mike McCarthy,John Fox,Craig Wrolstad,CHI98,Soldier Field
4251,2015_01_KC_HOU,2015,REG,1,2015-09-13,Sunday,13:00,KC,27.0,HOU,...,NaN,00-0023436,00-0026625,Alex Smith,Brian Hoyer,Andy Reid,Bill O'Brien,Peter Morelli,HOU00,NRG Stadium
4252,2015_01_CAR_JAX,2015,REG,1,2015-09-13,Sunday,13:00,CAR,20.0,JAX,...,7.0,00-0027939,00-0031407,Cam Newton,Blake Bortles,Ron Rivera,Gus Bradley,Ron Torbert,JAX00,EverBank Field


In [3]:
print(schedules['season'].unique())
print(schedules.columns.tolist())
schedules.info()

[2015 2016 2017 2018 2019 2020 2021 2022 2023 2024 2025]
['game_id', 'season', 'game_type', 'week', 'gameday', 'weekday', 'gametime', 'away_team', 'away_score', 'home_team', 'home_score', 'location', 'result', 'total', 'overtime', 'old_game_id', 'gsis', 'nfl_detail_id', 'pfr', 'pff', 'espn', 'ftn', 'away_rest', 'home_rest', 'away_moneyline', 'home_moneyline', 'spread_line', 'away_spread_odds', 'home_spread_odds', 'total_line', 'under_odds', 'over_odds', 'div_game', 'roof', 'surface', 'temp', 'wind', 'away_qb_id', 'home_qb_id', 'away_qb_name', 'home_qb_name', 'away_coach', 'home_coach', 'referee', 'stadium_id', 'stadium']
<class 'pandas.core.frame.DataFrame'>
Int64Index: 3028 entries, 4248 to 7275
Data columns (total 46 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   game_id           3028 non-null   object 
 1   season            3028 non-null   int64  
 2   game_type         3028 non-null   object 
 3   week              302

In [4]:
import sqlite3

# This creates the database file
conn = sqlite3.connect('../data/nfl.db')
cursor = conn.cursor()

# Create the games table
cursor.execute('''
CREATE TABLE IF NOT EXISTS games (
    game_id TEXT PRIMARY KEY,
    season INTEGER,
    game_type TEXT,
    week INTEGER,
    gameday TEXT,
    weekday TEXT,
    gametime TEXT,
    home_team TEXT,
    away_team TEXT,
    home_score REAL,
    away_score REAL,
    result REAL,
    total REAL,
    overtime REAL,
    location TEXT,
    home_rest INTEGER,
    away_rest INTEGER,
    div_game INTEGER,
    roof TEXT,
    surface TEXT,
    temp REAL,
    wind REAL,
    referee TEXT,
    stadium_id TEXT,
    stadium TEXT
)
''')

# Create the betting_lines table
cursor.execute('''
CREATE TABLE IF NOT EXISTS betting_lines (
    game_id TEXT PRIMARY KEY,
    spread_line REAL,
    home_spread_odds REAL,
    away_spread_odds REAL,
    total_line REAL,
    over_odds REAL,
    under_odds REAL,
    home_moneyline REAL,
    away_moneyline REAL,
    FOREIGN KEY (game_id) REFERENCES games (game_id)
)
''')

# Create the game_personnel table
cursor.execute('''
CREATE TABLE IF NOT EXISTS game_personnel (
    game_id TEXT PRIMARY KEY,
    home_qb_id TEXT,
    home_qb_name TEXT,
    away_qb_id TEXT,
    away_qb_name TEXT,
    home_coach TEXT,
    away_coach TEXT,
    FOREIGN KEY (game_id) REFERENCES games (game_id)
)
''')

conn.commit()
print("Tables created successfully!")

Tables created successfully!


In [5]:
# Load into games table
games_df = schedules[[
    'game_id', 'season', 'game_type', 'week', 'gameday', 'weekday', 'gametime',
    'home_team', 'away_team', 'home_score', 'away_score', 'result', 'total',
    'overtime', 'location', 'home_rest', 'away_rest', 'div_game',
    'roof', 'surface', 'temp', 'wind', 'referee', 'stadium_id', 'stadium'
]]
games_df.to_sql('games', conn, if_exists='replace', index=False)

# Load into betting_lines table
betting_df = schedules[[
    'game_id', 'spread_line', 'home_spread_odds', 'away_spread_odds',
    'total_line', 'over_odds', 'under_odds', 'home_moneyline', 'away_moneyline'
]]
betting_df.to_sql('betting_lines', conn, if_exists='replace', index=False)

# Load into game_personnel table
personnel_df = schedules[[
    'game_id', 'home_qb_id', 'home_qb_name', 'away_qb_id', 'away_qb_name',
    'home_coach', 'away_coach'
]]
personnel_df.to_sql('game_personnel', conn, if_exists='replace', index=False)

conn.commit()
print("Data loaded successfully!")
print(f"Games loaded: {len(games_df)}")
betting_df.head()

Data loaded successfully!
Games loaded: 3028


,game_id,spread_line,home_spread_odds,away_spread_odds,total_line,over_odds,under_odds,home_moneyline,away_moneyline
4248,2015_01_PIT_NE,7.5,101.0,-111.0,51.0,-103.0,-107.0,-350.0,305.0
4249,2015_01_IND_BUF,-1.0,-107.0,-103.0,44.5,-113.0,102.0,-101.0,-109.0
4250,2015_01_GB_CHI,-5.5,100.0,-110.0,48.5,-103.0,-107.0,222.0,-250.0
4251,2015_01_KC_HOU,-1.0,-110.0,100.0,41.0,-107.0,-103.0,-105.0,-105.0
4252,2015_01_CAR_JAX,-3.0,-110.0,-100.0,41.0,-103.0,-107.0,131.0,-145.0
